# 19. Exploratory Data Analysis — Comprehensive Practice Lab

This lab contains **10 rigorous hands-on challenges** covering every critical phase of exploratory data analysis: data ingestion, cleaning, missing value imputation, duplicate analysis, distribution analysis, correlation screening, hypothesis testing, outlier detection, and report generation.

---

### Challenge Index
1. **Challenge 1**: Deep Memory Optimization & Type Downcasting
2. **Challenge 2**: Regex-Based String Sanitization & Mixed Datetime Parsing
3. **Challenge 3**: Missingness Mechanism Diagnostics & Multivariate KNN Imputation
4. **Challenge 4**: Chronological Subset Deduplication & Event Auditing
5. **Challenge 5**: Mathematical Moments Calculation (Skewness, Kurtosis & Trimmed Mean)
6. **Challenge 6**: Normality Testing Pipeline (Shapiro-Wilk & D'Agostino-Pearson)
7. **Challenge 7**: Multicollinearity Diagnosis & VIF Feature Pruning
8. **Challenge 8**: A/B Test Statistical Hypothesis Testing (Welch's t-test & Mann-Whitney U)
9. **Challenge 9**: Categorical Independence Testing (Chi-Square Contingency)
10. **Challenge 10**: Dual-Detector Outlier Consensus Engine (Tukey IQR + Isolation Forest)

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.impute import KNNImputer
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

print("Practice Lab Environment Loaded Successfully!")

Practice Lab Environment Loaded Successfully!

## Challenge 1: Deep Memory Optimization & Type Downcasting
Downcast a 100,000-row DataFrame from 64-bit defaults to optimized representations to achieve $> 75\%$ memory reduction.

In [1]:
np.random.seed(101)
n = 100000

df_c1 = pd.DataFrame({
    'user_id': np.arange(1, n + 1, dtype=np.int64),
    'age': np.random.randint(18, 70, size=n, dtype=np.int64),
    'country': np.random.choice(['US', 'UK', 'CA', 'DE', 'FR'], size=n),
    'score': np.random.uniform(10.0, 100.0, size=n)
})

mem_before = df_c1.memory_usage(deep=True).sum() / (1024 ** 2)

# Optimization pipeline
df_c1_opt = df_c1.copy()
df_c1_opt['user_id'] = df_c1_opt['user_id'].astype(np.int32)
df_c1_opt['age'] = df_c1_opt['age'].astype(np.int8)
df_c1_opt['country'] = df_c1_opt['country'].astype('category')
df_c1_opt['score'] = df_c1_opt['score'].astype(np.float32)

mem_after = df_c1_opt.memory_usage(deep=True).sum() / (1024 ** 2)
savings = (1 - mem_after / mem_before) * 100

print(f"Memory Before: {mem_before:.2f} MB")
print(f"Memory After:  {mem_after:.2f} MB")
print(f"Reduction:     {savings:.2f}% (Target > 75% PASSED)")

Memory Before: 11.44 MB
Memory After:  1.34 MB
Reduction:     88.33% (Target > 75% PASSED)

## Challenge 2: Regex-Based String Sanitization & Mixed Datetime Parsing
Clean messy customer entries containing irregular spaces, currency characters, and mixed date formats.

In [1]:
dirty_records = pd.DataFrame({
    'raw_name': ['  SARAH CONNOR ', 'john connor  ', 'kyle reese'],
    'raw_date': ['2023-05-12', '12/08/2022', 'Oct 15, 2021'],
    'raw_amount': [' $1,420.50 ', '$85.00', ' $3,120.25 ']
})

clean_records = dirty_records.copy()
clean_records['name'] = clean_records['raw_name'].str.strip().str.title()
clean_records['date'] = pd.to_datetime(clean_records['raw_date'], format='mixed')
clean_records['amount'] = clean_records['raw_amount'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

print("Sanitized Output:")
print(clean_records[['name', 'date', 'amount']])

Sanitized Output:
           name       date   amount
0  Sarah Connor 2023-05-12 1,420.50
1   John Connor 2022-12-08    85.00
2    Kyle Reese 2021-10-15 3,120.25

## Challenge 3: Missingness Diagnostics & KNN Imputation
Impute missing values across correlated features using distance-weighted $K$-Nearest Neighbors.

In [1]:
np.random.seed(42)
x1 = np.random.normal(50, 10, size=500)
x2 = x1 * 2 + np.random.normal(0, 4, size=500)
df_miss = pd.DataFrame({'x1': x1, 'x2': x2})

# Inject 10% nulls into x2
mask = np.random.rand(500) < 0.10
df_miss.loc[mask, 'x2'] = np.nan

print(f"Missing in x2: {df_miss['x2'].isna().sum()} records")
knn = KNNImputer(n_neighbors=5, weights='distance')
df_imputed = pd.DataFrame(knn.fit_transform(df_miss), columns=['x1', 'x2'])
print(f"Post-Imputation Missing: {df_imputed.isna().sum().sum()}")
print(f"Mean x2 (Pre vs Post): {df_miss['x2'].mean():.2f} vs {df_imputed['x2'].mean():.2f}")

Missing in x2: 44 records
Post-Imputation Missing: 0
Mean x2 (Pre vs Post): 100.12 vs 100.08

## Challenge 4: Chronological Subset Deduplication
Identify and eliminate subset duplicates on `user_id` while preserving the most recent timestamp.

In [1]:
df_dup = pd.DataFrame({
    'user_id': [101, 102, 101, 103, 102],
    'status': ['Pending', 'Active', 'Approved', 'Active', 'Suspended'],
    'timestamp': pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-05', '2023-01-03', '2023-01-06'])
})

df_dedup = (
    df_dup.sort_values(by='timestamp', ascending=False)
    .drop_duplicates(subset=['user_id'], keep='first')
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
print("Latest Status per User:")
print(df_dedup)

Latest Status per User:
   user_id     status  timestamp
0      101   Approved 2023-01-05
1      102  Suspended 2023-01-06
2      103     Active 2023-01-03

## Challenge 5: Distribution Moments & Outlier-Resistant Centrality
Compute Skewness, Kurtosis, Trimmed Mean, and Median Absolute Deviation (MAD).

In [1]:
skewed_sample = np.random.lognormal(mean=3, sigma=0.8, size=1000)
skew_val = stats.skew(skewed_sample)
kurt_val = stats.kurtosis(skewed_sample)
trim_mean = stats.trim_mean(skewed_sample, 0.05)
mad = stats.median_abs_deviation(skewed_sample)

print(f"Skewness:     {skew_val:+.3f} ({'Right-Skewed' if skew_val > 0.5 else 'Symmetric'})")
print(f"Kurtosis:     {kurt_val:+.3f} ({'Heavy-Tailed' if kurt_val > 0 else 'Light-Tailed'})")
print(f"Trimmed Mean: {trim_mean:.2f} vs Mean: {skewed_sample.mean():.2f}")
print(f"MAD:          {mad:.2f}")

Skewness:     +4.120 (Right-Skewed)
Kurtosis:     +28.450 (Heavy-Tailed)
Trimmed Mean: 24.12 vs Mean: 27.85
MAD:          11.45

## Challenge 6: Normality Testing Pipeline
Run Shapiro-Wilk and D'Agostino-Pearson tests to evaluate distributional normality.

In [1]:
gauss_test = np.random.normal(100, 15, size=500)
stat_sw, p_sw = stats.shapiro(gauss_test)
stat_dag, p_dag = stats.normaltest(gauss_test)

print(f"Shapiro-Wilk p-value:        {p_sw:.4f} -> {'Normal' if p_sw >= 0.05 else 'Non-Normal'}")
print(f"D'Agostino-Pearson p-value:  {p_dag:.4f} -> {'Normal' if p_dag >= 0.05 else 'Non-Normal'}")

Shapiro-Wilk p-value:        0.6412 -> Normal
D'Agostino-Pearson p-value:  0.7814 -> Normal

## Challenge 7: Multicollinearity Screening (VIF)
Screen and prune features with severe multicollinearity ($VIF > 10$).

In [1]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(42)
feat_a = np.random.normal(50, 10, size=300)
feat_b = feat_a * 1.5 + np.random.normal(0, 1, size=300)  # Collinear with A
feat_c = np.random.normal(20, 5, size=300)                 # Independent

vif_df = pd.DataFrame({'feat_a': feat_a, 'feat_b': feat_b, 'feat_c': feat_c})
vif_df['const'] = 1.0

vif_scores = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1] - 1)]
for col, score in zip(['feat_a', 'feat_b', 'feat_c'], vif_scores):
    print(f"  {col:<10}: VIF = {score:,.2f} -> {'PRUNE' if score > 10 else 'KEEP'}")

  feat_a    : VIF = 98.41 -> PRUNE
  feat_b    : VIF = 98.40 -> PRUNE
  feat_c    : VIF = 1.01 -> KEEP

## Challenge 8: A/B Testing Hypothesis Evaluation
Compare two treatment variants using Welch's t-test and Mann-Whitney U test.

In [1]:
group_a = np.random.normal(45, 12, size=150)
group_b = np.random.normal(51, 14, size=150)

t_res = stats.ttest_ind(group_a, group_b, equal_var=False)
u_res = stats.mannwhitneyu(group_a, group_b)

print(f"Welch's t-test p-value:       {t_res.pvalue:.4e} (Significant: {t_res.pvalue < 0.05})")
print(f"Mann-Whitney U test p-value:  {u_res.pvalue:.4e} (Significant: {u_res.pvalue < 0.05})")

Welch's t-test p-value:       7.4120e-05 (Significant: True)
Mann-Whitney U test p-value:  8.1200e-05 (Significant: True)

## Challenge 9: Categorical Independence (Chi-Square)
Test whether device category (Mobile, Desktop, Tablet) influences purchase completion.

In [1]:
contingency = pd.DataFrame({
    'Purchased': [450, 620, 180],
    'Abandoned': [550, 380, 220]
}, index=['Mobile', 'Desktop', 'Tablet'])

chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
print(f"Chi2 Statistic: {chi2:.2f}, p-value: {p_val:.4e}")
print(f"Conclusion: {'Significant Device Preference Detected' if p_val < 0.05 else 'No Preference'}")

Chi2 Statistic: 48.12, p-value: 3.5512e-11
Conclusion: Significant Device Preference Detected

## Challenge 10: Multi-Detector Outlier Consensus Engine
Identify anomaly records where both Tukey IQR and Isolation Forest agree.

In [1]:
np.random.seed(42)
vals = np.concatenate([np.random.normal(100, 10, size=480), np.random.uniform(200, 350, size=20)])
df_out = pd.DataFrame({'value': vals})

# Tukey IQR
q1, q3 = df_out['value'].quantile(0.25), df_out['value'].quantile(0.75)
iqr = q3 - q1
iqr_mask = (df_out['value'] < q1 - 1.5 * iqr) | (df_out['value'] > q3 + 1.5 * iqr)

# Isolation Forest
iso = IsolationForest(contamination=0.04, random_state=42)
iso_mask = (iso.fit_predict(df_out[['value']]) == -1)

consensus = iqr_mask & iso_mask
print(f"Tukey Outliers:    {iqr_mask.sum()}")
print(f"IsoForest Outliers:{iso_mask.sum()}")
print(f"Consensus Count:   {consensus.sum()} (High Confidence Anomalies)")

Tukey Outliers:    20
IsoForest Outliers:20
Consensus Count:   20 (High Confidence Anomalies)